In [ ]:
%%bash
set -e

# 1. Redirect all caches and model storage to /tmp to avoid the 20GB /kaggle/working limit
export HF_HOME=/tmp/huggingface
export TORCH_HOME=/tmp/torch
mkdir -p /tmp/huggingface /tmp/torch /tmp/outputs

# 2. Install Cloudflare Tunnel and FFmpeg
wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
dpkg -i cloudflared-linux-amd64.deb
apt-get update -qq && apt-get install -y -qq ffmpeg

# 3. Clone Wan2GP into /tmp
cd /tmp
git clone https://github.com/deepbeepmeep/Wan2GP.git
cd Wan2GP

# 4. Install dependencies compatible with Kaggle's pre-installed PyTorch/CUDA
pip install -q -r requirements.txt
pip install -q optimum-quanto accelerate safetensors huggingface_hub[cli]

# 5. Symlink checkpoints and outputs cleanly
mkdir -p /tmp/ckpts /kaggle/working/wan22_videos
rm -rf /tmp/Wan2GP/ckpts /tmp/Wan2GP/outputs
ln -s /tmp/ckpts /tmp/Wan2GP/ckpts
ln -s /kaggle/working/wan22_videos /tmp/Wan2GP/outputs

echo "Setup complete. Available scratch disk space:"
df -h /tmp

In [ ]:
import os
import re
import subprocess
import time

# 1. Switch working directory into the repo so relative paths (models/_settings.json) resolve
os.chdir("/tmp/Wan2GP")

os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 2. Start Cloudflare Tunnel pointing to Wan2GP's default port (7860)
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start_time = time.time()
while time.time() - start_time < 25:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n==========================================")
print(f"Wan 2.2 Web UI URL: {public_url}")
print("==========================================\n")
print("Wait ~15 seconds for the server to finish booting before opening the link...")

# 3. Launch Wan2GP from inside /tmp/Wan2GP
!python wgp.py --profile 4 --fp16 --attention sdpa --server-port 7860